In [2]:
import pandas as pd
import numpy as np
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# ---------- 1. Read embeddings ----------
emb_path = "./embedding_MolCLR.csv"
emb_df = pd.read_csv(emb_path)
# Assuming column order: drug_id, label, embedding(string)
emb_df.columns = ['drug_id', 'label', 'embedding']  # Rename for convenience

# Convert string embeddings to numpy vectors
emb_df['vec'] = emb_df['embedding'].apply(lambda s: np.fromstring(s.strip('[]'), sep=','))
X = np.vstack(emb_df['vec'].values)  # (N, dim)

# ---------- 2. t-SNE dimensionality reduction ----------
tsne = TSNE(init='pca', n_components=2, random_state=42, perplexity=30, n_iter=5000, n_iter_without_progress=300)
emb_2d = tsne.fit_transform(X)
emb_df[['TSNE1', 'TSNE2']] = emb_2d

# ---------- 3. Extract contents from file segments ----------
# Content of file segment 1: rows 2 to 53 from emb_path file
file1_df = emb_df.iloc[1:53].copy()  # Rows 2 to 53
file1_ids = file1_df['drug_id'].unique()

# Content of file segment 2: rows 53 to 92 from emb_path file
file2_df = emb_df.iloc[53:92].copy()  # Rows 53 to 92
file2_ids = file2_df['drug_id'].unique()

# ---------- 4. Plot visualization ----------
plt.figure(figsize=(10, 8), dpi=300)
ax = plt.gca()

# All points: small grey dots
base = ax.scatter(emb_df['TSNE1'], emb_df['TSNE2'],
                  c='grey', s=2, alpha=0.3, label='Others')

# File segment 1: large blue dots (label=0) and large dark blue dots (label=1)
mask_file1 = emb_df['drug_id'].isin(file1_ids)  # Data from file segment 1
file1_colors = file1_df['label'].apply(lambda y: '#68baa6' if y == 0 else '#272396')
file1_scatter = ax.scatter(file1_df['TSNE1'], file1_df['TSNE2'], c=file1_colors, s=10, zorder=3)

# File segment 2: large purple dots (label=0) and large dark purple dots (label=1)
mask_file2 = emb_df['drug_id'].isin(file2_ids)  # Data from file segment 2
file2_colors = file2_df['label'].apply(lambda y: '#a668ba' if y == 0 else '#962327')
file2_scatter = ax.scatter(file2_df['TSNE1'], file2_df['TSNE2'], c=file2_colors, s=10, zorder=3)

# MAMPUM02: large red star
mask_mamp = (emb_df['drug_id'] == 'MAMPUM02')
if mask_mamp.any():
    mamp_scatter = ax.scatter(emb_df.loc[mask_mamp, 'TSNE1'],
                              emb_df.loc[mask_mamp, 'TSNE2'],
                              marker='*', c='red', s=100, edgecolor='k',
                              label='MAMPUM02', zorder=4)

# Aesthetics
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.set_title('t-SNE of MolCLR embeddings')

# Complete legend
from matplotlib.lines import Line2D

# Custom legend elements
legend_elements = [
    Line2D([0], [0], marker='o', color='w', label='Others', markerfacecolor='grey', markersize=5),
    Line2D([0], [0], marker='o', color='w', label='CA-based Cocrystals[1:1] (label=0)', markerfacecolor='#68baa6', markersize=8),
    Line2D([0], [0], marker='o', color='w', label='CA-based Cocrystals[1:1](label=1)', markerfacecolor='#272396', markersize=8),
    Line2D([0], [0], marker='o', color='w', label='CA-based Cocrystals[1:2](label=0)', markerfacecolor='#a668ba', markersize=8),
    Line2D([0], [0], marker='o', color='w', label='CA-based Cocrystals[1:2](label=1)', markerfacecolor='#962327', markersize=8),
    Line2D([0], [0], marker='*', color='w', label='MAMPUM02', markerfacecolor='red', markersize=10)
]

ax.legend(handles=legend_elements, loc='best')

plt.tight_layout()

# Save visualization
out_pdf = "./tsne_highlighted.pdf"
plt.savefig(out_pdf, format='pdf', bbox_inches='tight')
print(f'✅ Highlighted plot saved → {out_pdf}')
plt.show()

# ---------- 5. Save t-SNE embedding coordinates ----------
output_path = "./tsne_coordinates.csv"
emb_df[['drug_id', 'TSNE1', 'TSNE2']].to_csv(output_path, index=False)
print(f'✅ t-SNE embedding coordinates saved → {output_path}')
    


KeyboardInterrupt

